# TinyYOLO Experiment Suite — Notebook 2/7
## COCO val2017 Benchmark (Table IV)

**What:** Train TinyYOLO-q on COCO for the COCO SOTA comparison table.

**GPU Time:** ~24h T4 (3 seeds × ~8h each)

**Platform:** RunPod/Vast.ai recommended. For Kaggle/Colab, run 1 seed per session.

In [ ]:
#@title ⚙️ Configuration — Edit this cell

FAST_MODE = True  #@param {type:"boolean"}

if FAST_MODE:
    print("🚀 Running in FAST_MODE (Rapid Verification)")
    SEEDS = [42]                 # Single seed instead of 3
    EPOCHS = 100                 # Early convergence with pre-training
    IMGSZ = 320                  # 40% reduction in conv FLOPs
    BATCH = 32
    DATA = 'coco2017-subset.yaml' # Custom lightweight subset for resource limits
else:
    print("📚 Running in FULL PUBLICATION RIGOR mode")
    SEEDS = [42, 123, 256]
    EPOCHS = 300
    IMGSZ = 416
    BATCH = 32
    DATA = 'coco.yaml'

REPO = 'https://github.com/ShMazumder/tinyYOLO.git'

In [ ]:
import os, sys, socket, shutil, zipfile, yaml
from pathlib import Path
try: socket.create_connection(('github.com', 80), timeout=5)
except: print('❌ No internet!'); sys.exit(1)
if Path('tinyYOLO').exists(): shutil.rmtree('tinyYOLO')
if not Path('setup.py').exists():
    !git clone {REPO} --depth 1
    %cd tinyYOLO
    !pip install -e . -q 2>&1 | tail -1
    !pip install tqdm timm psutil -q 2>&1 | tail -1
else:
    print('✓ Running in cloned tinyYOLO directory')

import torch
if torch.cuda.is_available():
    print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')

# --- Setup and download COCO2017-subset if in FAST_MODE ---
if DATA == 'coco2017-subset.yaml':
    print('📂 Preparing COCO 2017 Subset...')
    base_path = Path('datasets/coco2017-subset')
    is_kaggle = Path('/kaggle').exists()
    
    if is_kaggle:
        # Check standard and nested Kaggle paths
        kaggle_input = None
        for p in [Path('/kaggle/input/coco2017-subset'), Path('/kaggle/input/datasets/abdelrahmanelgharibx/coco2017-subset')]:
            if p.exists():
                kaggle_input = p
                break
        if not kaggle_input:
            raise RuntimeError(
                '\n' + '='*80 + '\n' +
                '🛑 ERROR: Dataset "coco2017-subset" is not attached to this Kaggle notebook!\n' +
                'Downloading the 11.7GB dataset directly via Kaggle API will exceed the 20GB writable limit\n' +
                'under /kaggle/working during extraction, causing immediate kernel crash (disk limit exceeded).\n\n' +
                'Please follow these steps to attach the dataset as a read-only input (0% disk space penalty):\n' +
                '  1. On the right sidebar of the Kaggle notebook interface, click "+ Add Data"\n' +
                '  2. Search for: "coco2017-subset" (specifically by abdelrahmanelgharibx)\n' +
                '  3. Click "Add" next to "COCO 2017 Subset"\n' +
                '  4. Re-run this cell once the dataset is attached.\n' +
                '='*80
            )
        else:
            print(f'✓ Found attached Kaggle dataset at: {kaggle_input}')
            base_path = kaggle_input
    else:
        # Check if we have enough local disk space (need at least 25GB free to download & unzip safely)
        total, used, free = shutil.disk_usage('.')
        free_gb = free / (1024**3)
        print(f'Disk space check: {free_gb:.1f} GB free.')
        if free_gb < 25.0 and not (base_path / 'images').exists() and not Path('datasets/coco2017-subset-yolo').exists():
            print('⚠️ Local free space is less than 25 GB. Falling back to coco128.yaml to prevent running out of disk space.')
            DATA = 'coco128.yaml'
        else:
            # Try downloading if not already present
            if not base_path.exists():
                print('Dataset not found locally. Attempting to download from Kaggle...')
                os.makedirs('datasets', exist_ok=True)
                if shutil.which('kaggle'):
                    try:
                        !kaggle datasets download -d abdelrahmanelgharibx/coco2017-subset -p datasets --unzip
                        zip_file = Path('datasets/coco2017-subset.zip')
                        if zip_file.exists():
                            with zipfile.ZipFile(zip_file, 'r') as z:
                                z.extractall(base_path)
                            zip_file.unlink()
                    except Exception as e:
                        print(f'⚠️ Kaggle download failed: {e}. Falling back to coco128.yaml.')
                        DATA = 'coco128.yaml'
                else:
                    print('⚠️ Kaggle CLI not found. Falling back to coco128.yaml.')
                    DATA = 'coco128.yaml'
                    
    if DATA == 'coco2017-subset.yaml':
        # Dynamically build a writable YOLO-compliant directory structure using symlinks and converted labels
        import json
        target_dir = Path('datasets/coco2017-subset-yolo').absolute()
        target_dir.mkdir(parents=True, exist_ok=True)
        
        all_dirs = [base_path] + [p for p in base_path.rglob('*') if p.is_dir()]
        train_img_src = None
        val_img_src = None
        
        for d in all_dirs:
            has_imgs = any(d.glob('*.jpg')) or any(d.glob('*.png')) or any(d.glob('*.jpeg'))
            if has_imgs:
                name_lower = d.name.lower()
                if 'train2017' in name_lower or (('train' in name_lower) and not train_img_src):
                    train_img_src = d
                elif 'val2017' in name_lower or 'valid' in name_lower or (('val' in name_lower) and not val_img_src):
                    val_img_src = d
                    
        if not train_img_src or not val_img_src:
            # Fallback to any folder containing images
            for d in all_dirs:
                has_imgs = any(d.glob('*.jpg')) or any(d.glob('*.png')) or any(d.glob('*.jpeg'))
                if has_imgs:
                    train_img_src = d
                    val_img_src = d
                    break
                    
        if train_img_src and val_img_src:
            print(f'✓ Located Train images: {train_img_src}')
            print(f'✓ Located Val images: {val_img_src}')
            
            # Create symlinks for images
            images_target_dir = target_dir / 'images'
            images_target_dir.mkdir(parents=True, exist_ok=True)
            train_img_dst = images_target_dir / 'train2017'
            val_img_dst = images_target_dir / 'val2017'
            
            def safe_symlink(src, dst):
                src = Path(src).absolute()
                dst = Path(dst).absolute()
                if dst.exists() or dst.is_symlink():
                    if dst.is_symlink(): dst.unlink()
                    elif dst.is_dir(): shutil.rmtree(dst)
                    else: dst.unlink()
                try:
                    os.symlink(src, dst, target_is_directory=True)
                    print(f'✓ Created directory symlink: {dst} -> {src}')
                except Exception as e:
                    print(f'⚠️ Symlink failed: {e}. Copying images instead...')
                    shutil.copytree(src, dst)
            
            if train_img_src != val_img_src:
                safe_symlink(train_img_src, train_img_dst)
                safe_symlink(val_img_src, val_img_dst)
            else:
                print('✓ Train and Val images share the same source folder. Real target folders will be populated with individual symlinks during annotation conversion to avoid overlap.')
                train_img_dst.mkdir(parents=True, exist_ok=True)
                val_img_dst.mkdir(parents=True, exist_ok=True)
            
            # Create labels structure
            labels_target_dir = target_dir / 'labels'
            train_labels_dst = labels_target_dir / 'train2017'
            val_labels_dst = labels_target_dir / 'val2017'
            train_labels_dst.mkdir(parents=True, exist_ok=True)
            val_labels_dst.mkdir(parents=True, exist_ok=True)
            
            # Check for COCO json annotations
            train_json = None
            val_json = None
            for f in base_path.rglob('*.json'):
                if 'train' in f.name.lower():
                    train_json = f
                elif 'val' in f.name.lower() or 'valid' in f.name.lower():
                    val_json = f
                    
            if train_json and val_json:
                print(f'✓ Found COCO JSON annotations: train={train_json}, val={val_json}')
                print('Converting COCO annotations to YOLO format...')
                
                def convert_json(json_path, output_dir, img_src, img_dst, symlink_individual=False):
                    with open(json_path, 'r') as f_in:
                        data = json.load(f_in)
                    cats = sorted(data.get('categories', []), key=lambda x: x['id'])
                    cat_id_to_idx = {cat['id']: idx for idx, cat in enumerate(cats)}
                    images = {img['id']: img for img in data.get('images', [])}
                    
                    img_anns = {}
                    for ann in data.get('annotations', []):
                        img_id = ann['image_id']
                        if img_id not in images: continue
                        bbox = ann.get('bbox', [])
                        if len(bbox) != 4: continue
                        img_w = images[img_id]['width']
                        img_h = images[img_id]['height']
                        if img_w <= 0 or img_h <= 0: continue
                        
                        xmin, ymin, w, h = bbox
                        x_center = xmin + w / 2.0
                        y_center = ymin + h / 2.0
                        
                        x_norm = max(0.0, min(1.0, x_center / img_w))
                        y_norm = max(0.0, min(1.0, y_center / img_h))
                        w_norm = max(0.0, min(1.0, w / img_w))
                        h_norm = max(0.0, min(1.0, h / img_h))
                        
                        class_idx = cat_id_to_idx.get(ann['category_id'], ann['category_id'])
                        if img_id not in img_anns: img_anns[img_id] = []
                        img_anns[img_id].append(f'{class_idx} {x_norm:.6f} {y_norm:.6f} {w_norm:.6f} {h_norm:.6f}')
                        
                    converted_count = 0
                    for img_id, lines in img_anns.items():
                        file_name = images[img_id]['file_name']
                        src_img_file = Path(img_src) / file_name
                        if not src_img_file.exists():
                            continue
                        if symlink_individual:
                            dst_img_file = Path(img_dst) / file_name
                            if dst_img_file.exists() or dst_img_file.is_symlink():
                                if dst_img_file.is_symlink(): dst_img_file.unlink()
                                else: dst_img_file.unlink()
                            try:
                                os.symlink(src_img_file.absolute(), dst_img_file)
                            except Exception as e:
                                shutil.copy(src_img_file, dst_img_file)
                        txt_name = Path(file_name).with_suffix('.txt').name
                        with open(Path(output_dir) / txt_name, 'w') as f_out:
                            f_out.write('\n'.join(lines) + '\n')
                        converted_count += 1
                    print(f'✓ Converted {converted_count} annotations in {output_dir}')
                
                symlink_ind = (train_img_src == val_img_src)
                convert_json(train_json, train_labels_dst, train_img_src, train_img_dst, symlink_individual=symlink_ind)
                convert_json(val_json, val_labels_dst, val_img_src, val_img_dst, symlink_individual=symlink_ind)
            else:
                print('Checking for pre-existing YOLO labels in source...')
                train_lbl_src = None
                val_lbl_src = None
                for d in all_dirs:
                    if d.name.lower() == 'labels' or 'label' in d.name.lower():
                        for sub in d.iterdir():
                            if sub.is_dir():
                                if 'train' in sub.name.lower(): train_lbl_src = sub
                                elif 'val' in sub.name.lower() or 'valid' in sub.name.lower(): val_lbl_src = sub
                        if not train_lbl_src:
                            if 'train' in d.name.lower(): train_lbl_src = d
                            elif 'val' in d.name.lower() or 'valid' in d.name.lower(): val_lbl_src = d
                if train_lbl_src and val_lbl_src:
                    print(f'Copying pre-existing YOLO labels from {train_lbl_src} and {val_lbl_src}...')
                    for src_lbl, dst_lbl in [(train_lbl_src, train_labels_dst), (val_lbl_src, val_labels_dst)]:
                        for f_txt in src_lbl.glob('*.txt'):
                            shutil.copy(f_txt, dst_lbl)
                    print('✓ Copied labels successfully.')
                else:
                    print('⚠️ No COCO JSON annotations or pre-existing YOLO label directories found. Falling back to coco128.yaml.')
                    DATA = 'coco128.yaml'
            
            if DATA == 'coco2017-subset.yaml':
                # Read standard COCO names from coco128.yaml if it exists
                coco_names = {}
                if Path('datasets/coco128.yaml').exists():
                    with open('datasets/coco128.yaml', 'r') as f_coco:
                        coco_cfg = yaml.safe_load(f_coco)
                        coco_names = coco_cfg.get('names', {})
                else:
                    coco_names = {i: name for i, name in enumerate(['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'])}
                
                custom_cfg = {
                    'path': str(target_dir.absolute()),
                    'train': 'images/train2017',
                    'val': 'images/val2017',
                    'nc': 80,
                    'names': coco_names
                }
                
                os.makedirs('datasets', exist_ok=True)
                with open('datasets/coco2017-subset.yaml', 'w') as f_yml:
                    yaml.dump(custom_cfg, f_yml, default_flow_style=False)
                print('✓ Successfully generated datasets/coco2017-subset.yaml')
        else:
            print('⚠️ Could not locate train/val image folders in the dataset structure.')
            print('Falling back to coco128.yaml...')
            DATA = 'coco128.yaml'
else:
    print(f'📂 Downloading {DATA} dataset...')
    !python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('{DATA}')" 2>&1 | tail -3
print('✅ Setup complete!')

In [ ]:
import time
for i, seed in enumerate(SEEDS):
    name = f'coco-q-{IMGSZ}-seed{seed}'
    if Path(f'experiments/results/{name}/summary.json').exists():
        print(f'⏭️  {name} — already done'); continue
    print(f'\n{"="*60}\n🏃 [{i+1}/{len(SEEDS)}] {name}\n{"="*60}')
    t0 = time.time()
    !python scripts/train.py --task det --variant quantized --lr 2e-4 --amp --cache --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 150 \
        --data {DATA} --imgsz {IMGSZ} --epochs {EPOCHS} \
        --batch {BATCH} --seed {seed} --warmup 3 \
        --pretrained --compile --name {name}
    print(f'⏱️  {(time.time()-t0)/3600:.1f}h')
print('✅ COCO benchmark complete!')

In [ ]:
import json, glob
import numpy as np
maps = []
for d in sorted(glob.glob(f'experiments/results/coco-q-{IMGSZ}-seed*')):
    s = Path(d) / 'summary.json'
    if s.exists():
        with open(s) as f: data = json.load(f)
        m = data.get('best_map50', 0) * 100
        maps.append(m)
        print(f'  {Path(d).name}: {m:.1f}%')
if maps:
    print(f'\n  TinyYOLO-q COCO mAP@50: {np.mean(maps):.1f} ± {np.std(maps):.1f}%')